[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_6_8/13_tag_6_8_transfer_learning_head_ersetzen.ipynb)

# Tag 6-8 – 13: Transfer Learning mit CIFAR-10

Wir trainieren ein CNN zunächst auf **fünf CIFAR-10-Klassen**. Anschließend werden seine gelernten Bildmerkmale für die **anderen fünf Klassen** wiederverwendet – mit einem vollständig neuen Klassifikations-Head.

## Idee

| Phase | Klassen | Was wird gelernt? |
| --- | --- | --- |
| Vortraining | airplane, automobile, bird, cat, deer | Feature Extractor + erster 5-Klassen-Head |
| Transfer | dog, frog, horse, ship, truck | Der alte Head wird verworfen. Ein neuer 5-Klassen-Head wird auf wenigen Bildern trainiert. |

Die Klassen selbst werden **nicht** übertragen: Ein Logit für `cat` kann nicht zu `dog` umbenannt werden. Übertragen werden die allgemeinen Bildmerkmale in den Convolution-Schichten, etwa Kanten, Texturen und Formen.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import ConfusionMatrixDisplay
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['axes.grid'] = False

DATA_DIR = Path('datasets')
BATCH_SIZE = 64
SOURCE_IDS = [0, 1, 2, 3, 4]
TARGET_IDS = [5, 6, 7, 8, 9]
SOURCE_TRAIN_PER_CLASS, SOURCE_VALID_PER_CLASS = 3_000, 500
TARGET_TRAIN_PER_CLASS, TARGET_VALID_PER_CLASS = 250, 250

print('PyTorch:', torch.__version__)
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## CIFAR-10 aus dem lokalen Cache laden

Dieses Notebook lädt nichts aus dem Internet. Führe bei einem neuen Rechner zuerst Notebook 00 aus. Der offizielle Testsplit bleibt bis zur Schlussauswertung unangetastet.

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

try:
    cifar_train_aug = datasets.CIFAR10(DATA_DIR, train=True, download=False, transform=train_transform)
    cifar_train_eval = datasets.CIFAR10(DATA_DIR, train=True, download=False, transform=eval_transform)
    cifar_test = datasets.CIFAR10(DATA_DIR, train=False, download=False, transform=eval_transform)
except RuntimeError as error:
    raise FileNotFoundError(
        'CIFAR-10 wurde nicht im lokalen Cache gefunden. Bitte zuerst Notebook 00 ausführen.'
    ) from error

CLASS_NAMES = cifar_train_eval.classes
SOURCE_NAMES = [CLASS_NAMES[i] for i in SOURCE_IDS]
TARGET_NAMES = [CLASS_NAMES[i] for i in TARGET_IDS]
print('Vortraining:', SOURCE_NAMES)
print('Transfer:', TARGET_NAMES)

In [ ]:
def split_indices(labels, class_ids, train_per_class, valid_per_class, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    train_indices, valid_indices = [], []
    labels = np.asarray(labels)
    for class_id in class_ids:
        candidates = np.flatnonzero(labels == class_id)
        rng.shuffle(candidates)
        train_indices.extend(candidates[:train_per_class])
        valid_indices.extend(candidates[train_per_class:train_per_class + valid_per_class])
    return train_indices, valid_indices


class CifarClassSubset(Dataset):
    # Bild behalten, aber die ursprünglichen CIFAR-Labels auf 0 ... 4 abbilden.
    def __init__(self, dataset, indices, original_ids):
        self.dataset = dataset
        self.indices = list(indices)
        self.class_names = [CLASS_NAMES[i] for i in original_ids]
        self.label_map = {old_label: new_label for new_label, old_label in enumerate(original_ids)}

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        image, original_label = self.dataset[self.indices[position]]
        return image, torch.tensor(self.label_map[int(original_label)], dtype=torch.long)


def make_loader(dataset, shuffle=False):
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0,
                      generator=torch.Generator().manual_seed(RANDOM_STATE))

source_train_idx, source_valid_idx = split_indices(
    cifar_train_eval.targets, SOURCE_IDS, SOURCE_TRAIN_PER_CLASS, SOURCE_VALID_PER_CLASS
)
target_train_idx, target_valid_idx = split_indices(
    cifar_train_eval.targets, TARGET_IDS, TARGET_TRAIN_PER_CLASS, TARGET_VALID_PER_CLASS, seed=99
)
target_test_idx = [i for i, label in enumerate(cifar_test.targets) if label in TARGET_IDS]

source_train = CifarClassSubset(cifar_train_aug, source_train_idx, SOURCE_IDS)
source_valid = CifarClassSubset(cifar_train_eval, source_valid_idx, SOURCE_IDS)
target_train = CifarClassSubset(cifar_train_aug, target_train_idx, TARGET_IDS)
target_valid = CifarClassSubset(cifar_train_eval, target_valid_idx, TARGET_IDS)
target_test = CifarClassSubset(cifar_test, target_test_idx, TARGET_IDS)

source_train_loader = make_loader(source_train, shuffle=True)
source_valid_loader = make_loader(source_valid)
target_train_loader = make_loader(target_train, shuffle=True)
target_valid_loader = make_loader(target_valid)
target_test_loader = make_loader(target_test)

print(f'Vortraining: {len(source_train):,} Train / {len(source_valid):,} Validierung')
print(f'Transfer:    {len(target_train):,} Train / {len(target_valid):,} Validierung / {len(target_test):,} Test')

In [ ]:
# Die Bilder sind für das Modell normalisiert. Für die Anzeige machen wir das rückgängig.
MEAN = torch.tensor((0.4914, 0.4822, 0.4465)).view(3, 1, 1)
STD = torch.tensor((0.2470, 0.2435, 0.2616)).view(3, 1, 1)

def show_image(image):
    return (image.cpu() * STD + MEAN).permute(1, 2, 0).clamp(0, 1)

fig, axes = plt.subplots(2, 5, figsize=(11, 4.5))
for column in range(5):
    image, label = source_train[column * SOURCE_TRAIN_PER_CLASS]
    axes[0, column].imshow(show_image(image))
    axes[0, column].set_title(source_train.class_names[int(label)])
    axes[0, column].axis('off')

    image, label = target_train[column * TARGET_TRAIN_PER_CLASS]
    axes[1, column].imshow(show_image(image))
    axes[1, column].set_title(target_train.class_names[int(label)])
    axes[1, column].axis('off')
axes[0, 0].set_ylabel('Vortraining', fontsize=11)
axes[1, 0].set_ylabel('Transfer', fontsize=11)
plt.tight_layout()

## CNN und Trainingsfunktionen

`features` enthält den übertragbaren Bild-Extractor. `classifier` ist bewusst ein einzelner, klar ersetzbarer Head.

In [ ]:
class FeatureCNN(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        return self.classifier(self.features(x))


def train_epoch(model, data_loader, loss_fn, optimizer):
    model.train()
    total_loss = correct = seen = 0
    for images, labels in data_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        seen += len(labels)
    return total_loss / seen, correct / seen


def evaluate(model, data_loader, loss_fn):
    model.eval()
    total_loss = correct = seen = 0
    all_true, all_pred = [], []
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            total_loss += loss_fn(logits, labels).item() * len(labels)
            predictions = logits.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            seen += len(labels)
            all_true.append(labels.cpu())
            all_pred.append(predictions.cpu())
    return total_loss / seen, correct / seen, torch.cat(all_true), torch.cat(all_pred)


def train_epochs(model, train_loader, valid_loader, optimizer, epochs):
    history = []
    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, loss_fn, optimizer)
        valid_loss, valid_acc, _, _ = evaluate(model, valid_loader, loss_fn)
        history.append({'epoch': epoch + 1, 'train_acc': train_acc, 'valid_acc': valid_acc})
        print(f'Epoch {epoch + 1:02d}: train={train_acc:.2%}, valid={valid_acc:.2%}')
    return history


def count_trainable_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

loss_fn = nn.CrossEntropyLoss()

## 1. Vortraining auf den ersten fünf Klassen

Hier lernen sowohl Extractor als auch der erste Head. Danach wird dieser Head nicht weiterverwendet.

In [ ]:
torch.manual_seed(RANDOM_STATE)
base_model = FeatureCNN(num_classes=len(SOURCE_IDS)).to(device)
base_optimizer = torch.optim.Adam(base_model.parameters(), lr=1e-3, weight_decay=1e-4)
print('Trainierbare Parameter:', f'{count_trainable_parameters(base_model):,}')
pretrain_history = train_epochs(
    base_model, source_train_loader, source_valid_loader, base_optimizer, epochs=8
)

## 2. Neuer Head für die anderen fünf Klassen

Wir kopieren ausschließlich die Gewichte von `features`. Der alte Head für airplane bis deer wird verworfen und durch eine frisch initialisierte Schicht für dog bis truck ersetzt. Während des ersten Transfer-Schritts bleiben die Features eingefroren.

In [ ]:
torch.manual_seed(RANDOM_STATE)
scratch = FeatureCNN(num_classes=len(TARGET_IDS)).to(device)

transfer = FeatureCNN(num_classes=len(SOURCE_IDS)).to(device)
transfer.features.load_state_dict(base_model.features.state_dict())
transfer.classifier = nn.Linear(128, len(TARGET_IDS)).to(device)  # neuer, zufälliger Head
for parameter in transfer.features.parameters():
    parameter.requires_grad = False

print('Scratch – trainierbare Parameter: ', f'{count_trainable_parameters(scratch):,}')
print('Transfer – trainierbare Parameter:', f'{count_trainable_parameters(transfer):,}')
print('Neuer Transfer-Head:', transfer.classifier)

## 3. Fairer Vergleich: von Scratch oder mit Transfer

Beide Modelle sehen dieselben 250 Trainingsbilder je Zielklasse. Der Scratch-Vergleich zeigt, ob die vortrainierten Merkmale beim kleinen Zieltrainingssatz helfen.

In [ ]:
histories = {}
for name, model, optimizer in [
    ('Scratch', scratch, torch.optim.Adam(scratch.parameters(), lr=1e-3, weight_decay=1e-4)),
    ('Transfer: nur Head', transfer, torch.optim.Adam(transfer.classifier.parameters(), lr=3e-3)),
]:
    print(f'\n{name}')
    histories[name] = train_epochs(model, target_train_loader, target_valid_loader, optimizer, epochs=12)

for name, history in histories.items():
    plt.plot([row['valid_acc'] for row in history], marker='o', label=name)
plt.xlabel('Epoche')
plt.ylabel('Validierungs-Accuracy')
plt.ylim(0, 1)
plt.legend()
plt.show()

## 4. Optional: vorsichtiges Fine-Tuning

Nun wird nur der letzte Convolution-Block freigegeben. Die kleine Lernrate schützt die bereits gelernten allgemeinen Merkmale davor, mit dem kleinen Zieldatensatz sofort überschrieben zu werden.

In [ ]:
for parameter in transfer.features[6:].parameters():
    parameter.requires_grad = True

fine_tune_optimizer = torch.optim.Adam(
    (parameter for parameter in transfer.parameters() if parameter.requires_grad), lr=2e-4
)
print('Trainierbare Parameter nach Freigabe:', f'{count_trainable_parameters(transfer):,}')
fine_tune_history = train_epochs(
    transfer, target_train_loader, target_valid_loader, fine_tune_optimizer, epochs=4
)

## Schlussauswertung auf dem unberührten Testsplit

Die Testdaten wurden weder für die Modellwahl noch für Fine-Tuning verwendet.

In [ ]:
test_loss, test_acc, test_true, test_pred = evaluate(transfer, target_test_loader, loss_fn)
print(f'Test-Accuracy nach Transfer und Fine-Tuning: {test_acc:.2%}')

ConfusionMatrixDisplay.from_predictions(
    test_true.numpy(), test_pred.numpy(), display_labels=TARGET_NAMES, cmap='Blues', xticks_rotation=45
)
plt.title('CIFAR-10: andere fünf Klassen')
plt.grid(False)
plt.show()

images, labels = next(iter(target_test_loader))
with torch.no_grad():
    probabilities = torch.softmax(transfer(images.to(device)).cpu(), dim=1)
predictions = probabilities.argmax(dim=1)

fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for ax, image, true, prediction, confidence in zip(
    axes.ravel(), images[:12], labels[:12], predictions[:12], probabilities.max(dim=1).values[:12]
):
    correct = int(true) == int(prediction)
    ax.imshow(show_image(image))
    ax.set_title(
        f't: {TARGET_NAMES[int(true)]}\np: {TARGET_NAMES[int(prediction)]} ({float(confidence):.0%})',
        color='green' if correct else 'crimson', fontsize=9
    )
    ax.axis('off')
plt.tight_layout()

# 5. Praxisvariante: online vortrainiertes ResNet-18

Der vorherige Teil hat gezeigt, wie Merkmale von fünf CIFAR-10-Klassen auf fünf andere Klassen übertragen werden. In der Praxis beginnt man oft noch früher: mit einem Modell, das bereits auf einem großen externen Datensatz vortrainiert wurde.

Hier laden wir deshalb einmalig ein **ImageNet-vortrainiertes ResNet-18** aus dem Internet. Die Gewichte werden danach im PyTorch-Cache gespeichert. Auch hier wird sein ursprünglicher 1.000-Klassen-Head verworfen und durch einen neuen Head für unsere fünf CIFAR-Zielklassen ersetzt.

In [ ]:
from torchvision.models import ResNet18_Weights, resnet18

# Die ImageNet-Gewichte wurden mit 224 x 224 Bildern und dieser Normalisierung trainiert.
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
resnet_train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
resnet_eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Dieselben CIFAR-Bilder wie oben, aber im Eingabeformat des vortrainierten ResNet.
cifar_train_resnet = datasets.CIFAR10(DATA_DIR, train=True, download=False, transform=resnet_train_transform)
cifar_valid_resnet = datasets.CIFAR10(DATA_DIR, train=True, download=False, transform=resnet_eval_transform)
cifar_test_resnet = datasets.CIFAR10(DATA_DIR, train=False, download=False, transform=resnet_eval_transform)
resnet_target_train = CifarClassSubset(cifar_train_resnet, target_train_idx, TARGET_IDS)
resnet_target_valid = CifarClassSubset(cifar_valid_resnet, target_valid_idx, TARGET_IDS)
resnet_target_test = CifarClassSubset(cifar_test_resnet, target_test_idx, TARGET_IDS)

RESNET_BATCH_SIZE = 32
def resnet_loader(dataset, shuffle=False):
    return DataLoader(dataset, batch_size=RESNET_BATCH_SIZE, shuffle=shuffle, num_workers=0,
                      generator=torch.Generator().manual_seed(RANDOM_STATE))

resnet_train_loader = resnet_loader(resnet_target_train, shuffle=True)
resnet_valid_loader = resnet_loader(resnet_target_valid)
resnet_test_loader = resnet_loader(resnet_target_test)

weights = ResNet18_Weights.DEFAULT
try:
    online_resnet = resnet18(weights=weights, progress=True)
except Exception as error:
    raise RuntimeError(
        'Die ResNet-18-Gewichte konnten nicht geladen werden. Internetzugang erlauben und die Zelle erneut ausführen.'
    ) from error

# Alle ImageNet-Merkmale einfrieren; nur der neue CIFAR-Head darf zunächst lernen.
for parameter in online_resnet.parameters():
    parameter.requires_grad = False
online_resnet.fc = nn.Linear(online_resnet.fc.in_features, len(TARGET_IDS))
online_resnet = online_resnet.to(device)

print('Geladene Gewichte:', weights)
print('Neuer Head:', online_resnet.fc)
print('Trainierbare Parameter:', f'{count_trainable_parameters(online_resnet):,}')

## ResNet-18: nur den neuen Head trainieren

Die allgemeine Bildrepräsentation stammt jetzt aus ImageNet; deshalb genügt oft wenig CIFAR-Training für den neuen Head. Das Ergebnis ist ein zweites, unabhängiges Transfer-Learning-Beispiel neben dem 5-zu-5-Transfer oben. Beim ersten Ausführen wird das ResNet-Gewichtspaket einmalig online heruntergeladen.

In [ ]:
resnet_head_optimizer = torch.optim.Adam(online_resnet.fc.parameters(), lr=3e-3)
resnet_head_history = train_epochs(
    online_resnet, resnet_train_loader, resnet_valid_loader, resnet_head_optimizer, epochs=6
)

plt.plot([row['valid_acc'] for row in resnet_head_history], marker='o', label='ResNet-18: neuer Head')
plt.xlabel('Epoche')
plt.ylabel('Validierungs-Accuracy')
plt.ylim(0, 1)
plt.legend()
plt.show()

## Optional: letzten ResNet-Block feinjustieren

Wenn die Validierung noch steigt, kann zusätzlich der letzte Residual-Block `layer4` mit sehr kleiner Lernrate angepasst werden. Das ist deutlich günstiger als das ganze ResNet neu zu trainieren.

In [ ]:
for parameter in online_resnet.layer4.parameters():
    parameter.requires_grad = True

resnet_fine_tune_optimizer = torch.optim.Adam(
    (parameter for parameter in online_resnet.parameters() if parameter.requires_grad), lr=1e-4
)
resnet_fine_tune_history = train_epochs(
    online_resnet, resnet_train_loader, resnet_valid_loader, resnet_fine_tune_optimizer, epochs=3
)

_, resnet_test_acc, resnet_test_true, resnet_test_pred = evaluate(
    online_resnet, resnet_test_loader, loss_fn
)
print(f'ResNet-18 Test-Accuracy: {resnet_test_acc:.2%}')
ConfusionMatrixDisplay.from_predictions(
    resnet_test_true.numpy(), resnet_test_pred.numpy(),
    display_labels=TARGET_NAMES, cmap='Purples', xticks_rotation=45
)
plt.title('Online vortrainiertes ResNet-18: andere fünf CIFAR-10-Klassen')
plt.grid(False)
plt.show()